# Kolokvijum I rešen mehanizmima sa Kolokvijuma II

Isti zadatak kao u `k1/k1_resenje.ipynb` — **GlavnaKnjiga**, `pretraga`, `opoziv` — ali svuda gde je moguće
umesto osnovnog rešenja stoji mehanizam sa K2: generator, iterabilnost, transdjuseri i `Promise`.

Rešenje ispunjava **sve** zahteve iz teksta K1 zadatka; razlika je isključivo u sredstvima.

| Zahtev iz K1 | U `k1/k1_resenje.ipynb` | Ovde |
| --- | --- | --- |
| privatni statički redni broj | promenljiva u zatvorenju, `n++` | **beskonačan generator** u zatvorenju |
| pristup transakcijama | geter koji vraća kopiju niza | **`Symbol.iterator`** — knjiga je iterabilna kolekcija |
| istorija verzija | petlja koja unazad prati `prethodna` | **generator `verzije()`** — lenj obilazak lanca |
| izvedeno stanje | jedan `reduce` sa uslovom u telu | **kompozicija transdjusera** `filter` + `map` |
| `pretraga` | `reduce` preko kriterijuma, `n` filtera = `n` prolaza | **kompozicija transdjusera** — jedan prolaz |
| `opoziv` | `flatMap` preko niza verzija | **generator** koji propušta transakcije kasnijih verzija |
| knjiženje | ne postoji u K1 | **`Promise`**, `all` i `race` |

Struktura je vredna pažnje: pošto svaka knjiga drži referencu na prethodnu, **istorija je povezana lista** —
tačno struktura iz trećeg zadatka na K2, samo što je ovde dobijena zahtevom iz K1.

## Pravila funkcionalne paradigme kojih se rešenje drži

Ista pravila kao u `k1/k1_resenje.ipynb`, uz mehanizme sa K2:

| Pravilo | Kako je sprovedeno |
| --- | --- |
| **nepromenljivost** | `Object.freeze` nad transakcijom, spiskom i knjigom; svaka izmena daje novu verziju |
| **bez `this` i bez `new`** | fabričke funkcije; metode su strelice nad zatvorenjem — izuzetak su `[Symbol.iterator]` i `verzije`, koje jezik traži kao generator metode |
| **objekat se gradi jednim izrazom** | `Object.freeze(Object.defineProperties({ … }, { … }))` |
| **preklapanje i rekurzija umesto petlje** | `istorija` i `lanac` su rekurzivni, `uzmi` je rekurzivno; nigde `for` petlje ni brojača petlje |
| **lenjost** | generatori `verzije()` i tok kasnijih transakcija posećuju samo ono što se zaista pročita |
| **kompozicija** | `stanje` i `pretraga` su kompozicije transdjusera — jedan prolaz bez obzira na broj koraka |
| **efekti na ivici** | `console.log` samo u demonstracijama; `Promise` sloj stoji **pored** tipa (`knjizenje(knjiga)`), ne u njemu |

**Izuzetak, zahtevan tekstom zadatka:** redni broj transakcije. Ovde je stanje zatvoreno u **generatoru** (jedinica 1) —
i dalje stanje, ali sa jedinim mogućim pristupom `next()`, bez promenljive koju bi neko mogao da postavi.

---
# Jedinica 1 · Redni broj kao beskonačan generator

In [ ]:
function* brojac(od = 1) { let n = od; while (true) yield n++; }     // beskonačan tok — nikad se ne spreaduje

const STATUSI = Object.freeze(["nerealizovana", "realizovana", "stornirana"]);
const TIPOVI  = Object.freeze(["na teret", "u korist"]);

const Transakcija = (() => {
    const redniBrojevi = brojac();                                   // privatni statički brojač — tok u zatvorenju

    return (brojRacuna, opis, status, datum, tip, iznos) => {
        if (!STATUSI.includes(status)) throw new Error(`Nedozvoljen status: ${status}`);
        if (!TIPOVI.includes(tip))     throw new Error(`Nedozvoljen tip: ${tip}`);

        return Object.freeze({
            redniBroj: redniBrojevi.next().value,                    // sledeća vrednost toka
            brojRacuna, opis, status, datum, tip, iznos
        });
    };
})();

const stornirana = t => Object.freeze({ ...t, status: "stornirana" });

const t1 = Transakcija("265-0001", "Uplata kupca", "realizovana", "2026-08-01", "u korist", 120000);
const t2 = Transakcija("265-0001", "Zakup",        "realizovana", "2026-08-03", "na teret",  45000);

console.log("redni brojevi:", t1.redniBroj, t2.redniBroj);
console.log("brojač spolja:", typeof redniBrojevi);                  // undefined — ostao je u zatvorenju

### Šta ovde donosi mehanizam sa K2

Generator je **tok vrednosti sa sopstvenim stanjem**: `n` živi unutar njega, a spoljni kod ne može da ga ni pročita
ni postavi — jedino što može jeste da zatraži sledeću vrednost. Time je „privatni statički atribut" opisan tipom
podatka, a ne dogovorom da se promenljiva ne dira.

Praktična razlika u odnosu na `let n = 1; n++`: isti `brojac` se koristi za bilo koji drugi niz brojeva
(`brojac(1000)` za drugu firmu), i može se, po potrebi, zameniti tokom koji čita brojeve odnekud drugde —
`Transakcija` pri tome ostaje nepromenjena, jer je za nju bitno samo da postoji `next()`.

---
# Jedinica 2 · Alat: transdjuseri

In [ ]:
const collect = (acc, e) => [...acc, e];
const sum     = (acc, e) => acc + e;
const map     = f => next => (acc, e) => next(acc, f(e));
const filter  = p => next => (acc, e) => p(e) ? next(acc, e) : acc;
const compose = (...fs) => x => fs.reduceRight((acc, f) => f(acc), x);

// prvih n vrednosti iz toka — rekurzivno, bez menjanja niza; beskonačan tok se nikad ne spreaduje
const uzmi = (n, tok) => n <= 0 ? []
    : (({ value, done }) => done ? [] : [value, ...uzmi(n - 1, tok)])(tok.next());

console.log("alat spreman:", [1, 2, 3, 4].reduce(compose(filter(x => x > 2), map(x => x * 10))(collect), []));
console.log("uzmi iz beskonačnog toka:", uzmi(5, brojac(1)));

### Šta ovde donosi mehanizam sa K2

Transdjuseri razdvajaju **šta se radi sa elementom** od **odakle elementi dolaze** i **u šta se skupljaju**.
Zbog toga isti `filter`/`map` u nastavku rade i nad nizom transakcija, i nad iterabilnom knjigom, i nad generatorom
koji propušta transakcije kasnijih verzija — bez ijedne izmene.

Druga korist je broj prolaza: `n` kriterijuma pretrage spojenih `compose`-om obradi svaku transakciju jednom,
umesto da svaki napravi svoj međuniz.

---
# Jedinica 3 · `GlavnaKnjiga` — iterabilna kolekcija sa lancem verzija

**Pretpostavka.** „Redni broj glavne knjige u istoriji" je indeks u nizu verzija gde je `0` prva knjiga.
Generator `verzije()` prolazi lanac od **najnovije** ka najstarijoj, pa `istorija()` taj niz obrne.

In [ ]:
const potpisanIznos = t => t.tip === "u korist" ? t.iznos : -t.iznos;

const xformStanje = compose(                                   // stanje = suma realizovanih sa predznakom po tipu
    filter(t => t.status === "realizovana"),
    map(potpisanIznos)
);

function* lanac(k) { if (k !== null) { yield k; yield* lanac(k.prethodna); } }   // rekurzivan, lenj tok verzija

const GlavnaKnjiga = (naziv, maticniBroj, pib, transakcije = [], prethodna = null) => {
    const t = Object.freeze([...transakcije]);                     // privatna, zamrznuta kopija

    const knjiga = Object.freeze(Object.defineProperties({          // ceo objekat nastaje JEDNIM izrazom
        naziv, maticniBroj, pib, prethodna,

        *[Symbol.iterator]() { yield* t; },                        // knjiga JE kolekcija transakcija
        *verzije() { yield* lanac(knjiga); },                      // lanac verzija, od najnovije, lenjo

        dodajTransakciju:  tr        => GlavnaKnjiga(naziv, maticniBroj, pib, [...t, tr], knjiga),
        ukloniTransakciju: redniBroj => GlavnaKnjiga(naziv, maticniBroj, pib,
                                                     t.filter(x => x.redniBroj !== redniBroj), knjiga)
    }, {
        transakcije: { get: () => [...knjiga], enumerable: true },                    // kopija iz iteratora
        stanje:      { get: () => t.reduce(xformStanje(sum), 0), enumerable: true }   // jedan prolaz
    }));

    return knjiga;
};

// ceo lanac kao niz, od najstarije (indeks 0) — rekurzija umesto obrtanja niza
const istorija = knjiga => knjiga === null ? [] : [...istorija(knjiga.prethodna), knjiga];

const knjiga0 = GlavnaKnjiga("Merkur doo", "21456789", "108456789");
const knjiga1 = knjiga0.dodajTransakciju(t1);
const knjiga2 = knjiga1.dodajTransakciju(t2);

for (const t of knjiga2) console.log("  for...of:", t.opis, t.iznos);
console.log("spread:", [...knjiga2].length, "| stanje:", knjiga2.stanje);
console.log("stanja po verzijama:", istorija(knjiga2).map(k => k.stanje));
console.log("original netaknut:", knjiga0.transakcije.length, "| deljena transakcija:", [...knjiga1][0] === [...knjiga2][0]);

### Šta ovde donosi mehanizam sa K2

**Iterabilnost.** Umesto getera koji vraća kopiju niza, knjiga dobija `Symbol.iterator`, pa nad njom rade `for...of`,
spread, destrukturiranje i `yield*`. Geter `transakcije` je i dalje tu (zadatak ga traži), ali je sveden na `[...o]` —
kopiju pravi sam iterator.

**Generator `verzije()`.** Obilazak lanca je opisan jednom, na mestu gde lanac i nastaje, i **lenj** je: ako treba samo
poslednjih nekoliko verzija, ostatak se nikad ne poseti. Petlja koja gradi ceo niz to ne može.

**Transdjuser za stanje.** `filter(realizovana)` pa `map(potpisanIznos)` čita se kao sama definicija iz zadatka —
„razlika suma realizovanih u korist i na teret" — a i dalje je **jedan prolaz**, jer se koraci spajaju `compose`-om
pre nego što `reduce` krene.

**Povezana lista.** `prethodna` + `verzije()` čine istoriju jednostruko povezanom listom sa lenjim obilaskom —
ista struktura koja se na K2 traži u trećem zadatku, ovde dobijena iz zahteva K1 zadatka.

---
# Jedinica 4 · `pretraga` — kompozicija kriterijuma u jednom prolazu

In [ ]:
const pretraga = (kriterijumi, knjiga) =>
    [...knjiga].reduce(compose(...kriterijumi.map(filter))(collect), []);   // svaki kriterijum -> filter transdjuser

const realizovane = t => t.status === "realizovana";
const uKorist     = t => t.tip === "u korist";
const naTeret     = t => t.tip === "na teret";
const preko       = granica => t => t.iznos > granica;
const uMesecu     = mesec   => t => t.datum.slice(0, 7) === mesec;

const knjigaP = ["Avans kupca|realizovana|u korist|2026-08-04|60000",
                 "Struja|realizovana|na teret|2026-08-06|18000",
                 "Nabavka robe|nerealizovana|na teret|2026-09-01|90000"]
    .map(red => red.split("|"))
    .reduce((k, [opis, status, tip, datum, iznos]) =>
        k.dodajTransakciju(Transakcija("265-0001", opis, status, datum, tip, Number(iznos))), knjiga2);

console.log("sve:", [...knjigaP].length);
console.log("realizovane u korist:", pretraga([realizovane, uKorist], knjigaP).map(x => x.opis));
console.log("na teret preko 20000:", pretraga([naTeret, preko(20000)], knjigaP).map(x => x.opis));
console.log("avgust, realizovane:", pretraga([uMesecu("2026-08"), realizovane], knjigaP).map(x => x.opis));
console.log("bez kriterijuma:", pretraga([], knjigaP).length);

// kriterijumi stižu kao LISTA, pa se lista može sastaviti i u toku rada — npr. iz popunjenih polja pretrage
const popunjeno = { status: "realizovana", tip: "na teret", minIznos: 10000 };
const izPolja = [
    ...(popunjeno.status   ? [t => t.status === popunjeno.status] : []),
    ...(popunjeno.tip      ? [t => t.tip === popunjeno.tip]       : []),
    ...(popunjeno.minIznos ? [preko(popunjeno.minIznos)]          : [])
];
console.log("lista sastavljena u toku rada:", pretraga(izPolja, knjigaP).map(x => x.opis));

// isti kriterijumi, drugi oblik rezultata — bez ponovnog pisanja pretrage
const zbirPretrage = (kriterijumi, knjiga) =>
    [...knjiga].reduce(compose(...kriterijumi.map(filter), map(potpisanIznos))(sum), 0);

console.log("zbir realizovanih:", zbirPretrage([realizovane], knjigaP));

### Šta ovde donosi mehanizam sa K2

Kriterijumi su i dalje obične predikatske funkcije i i dalje stižu kao **lista**, ali se ta lista **pretvara u
transdjusere** (`kriterijumi.map(filter)`) i spaja `compose`-om. Pošto je lista vrednost, može da se sastavi u toku
rada i tek onda spoji u jedan prolaz. Posledice:

- **jedan prolaz** umesto `n` — nema međunizova, bez obzira na broj kriterijuma;
- **redosled je očuvan** doslovno: prvi kriterijum prvi vidi transakciju, kao što zadatak traži;
- **oblik rezultata je odvojen** od kriterijuma — `collect` daje niz, `sum` daje zbir, uz isti spisak kriterijuma
  (`zbirPretrage`), što sa lančanim `filter`-ima traži pisanje nove funkcije.

Cena je čitljivost: `compose(...kriterijumi.map(filter))(collect)` traži poznavanje transdjusera, dok
`kriterijumi.reduce((niz, k) => niz.filter(k), …)` razume svako ko zna `filter`.

---
# Jedinica 5 · `opoziv` — generator preko kasnijih verzija

In [ ]:
const opoziv = (knjiga, redniBrojUIstoriji) => {
    const verzije = istorija(knjiga);
    const stara = verzije[redniBrojUIstoriji];
    if (!stara) throw new Error(`Ne postoji verzija sa indeksom ${redniBrojUIstoriji}`);

    const bileRanije = new Set([...stara].map(x => x.redniBroj));

    function* kasnijeTransakcije() {                                   // lenj tok kroz sve kasnije verzije
        for (const v of verzije.slice(redniBrojUIstoriji + 1)) yield* v;
    }

    const nastaleKasnije = [...kasnijeTransakcije()]
        .filter(x => !bileRanije.has(x.redniBroj))                                     // samo nove
        .filter((x, i, niz) => niz.findIndex(y => y.redniBroj === x.redniBroj) === i)   // po jedan primerak
        .map(stornirana);                                                              // kopije, originali netaknuti

    return GlavnaKnjiga(knjiga.naziv, knjiga.maticniBroj, knjiga.pib,
                        [...stara, ...nastaleKasnije], knjiga);
};

const opozvana = opoziv(knjigaP, 2);

console.log("verzija 2 je imala:", [...istorija(knjigaP)[2]].map(x => x.opis));
console.log("opozvana:", [...opozvana].map(x => `${x.opis}(${x.status})`));
console.log("stanje pre:", knjigaP.stanje, "| posle:", opozvana.stanje);
console.log("istorija:", istorija(opozvana).length, "verzija");
console.log("ništa nije mutirano:", [...knjigaP].every(x => x.status !== "stornirana"));
try { opoziv(knjigaP, 99); } catch (g) { console.log("odbijeno:", g.message); }

### Šta ovde donosi mehanizam sa K2

**`yield* v` nad verzijom.** Pošto je knjiga iterabilna, generator `kasnijeTransakcije` prolazi kroz sve kasnije
verzije jednim redom koda — nema `flatMap(k => k.transakcije)` i nema pravljenja privremenog niza po verziji.

**`[...stara]`.** Transakcije stare verzije uzimaju se preko njenog iteratora, pa `opoziv` ne mora da zna kako knjiga
iznutra čuva spisak.

**`Set` se pravi jednom i samo se čita.** `bileRanije` nastaje iz stare verzije i posle toga se nijednom ne menja,
pa provera „ova je već postojala" ostaje čista, a i dalje je u konstantnom vremenu.

**Uklanjanje ponavljanja bez pomoćnog stanja.** Ista transakcija se javlja u svakoj kasnijoj verziji; zadržava se
ono pojavljivanje čiji je indeks jednak prvom nađenom (`findIndex`), pa nema akumulatora koji se usput puni.

---
# Jedinica 6 · `Promise` sloj — knjiženje uz potvrdu

**Pretpostavka.** K1 zadatak ne pominje asinhronost; ovaj sloj je dodatak koji pokazuje K2 mehanizam nad istim
podacima. Potvrda knjiženja simulira odgovor servera: traje neko vreme i može da otkaže.

In [ ]:
const PotvrdaGreska = redniBroj => new Error(`Knjiženje odbijeno za transakciju ${redniBroj}`);

const knjizenje = knjiga => Object.freeze({
    // potvrda za jednu transakciju — kašnjenje + dva razloga otkaza
    potvrdi(redniBroj) {
        return new Promise((resolve, reject) => setTimeout(() => {
            const nadjena = [...knjiga].find(x => x.redniBroj === redniBroj);
            if (!nadjena)                             reject(new Error(`Nema transakcije ${redniBroj}`));
            else if (nadjena.status === "stornirana") reject(PotvrdaGreska(redniBroj));
            else                                      resolve(nadjena);
        }, Math.random() * 200));
    },

    // uspeva samo ako SVE potvrde stignu -> tek tada nova verzija knjige
    async proknjizi(...redniBrojevi) {
        const potvrdjene = await Promise.all(redniBrojevi.map(b => knjizenje(knjiga).potvrdi(b)));
        const skup = new Set(potvrdjene.map(x => x.redniBroj));
        return GlavnaKnjiga(knjiga.naziv, knjiga.maticniBroj, knjiga.pib,
            [...knjiga].map(x => skup.has(x.redniBroj) ? Object.freeze({ ...x, status: "realizovana" }) : x),
            knjiga);
    },

    // prva potvrda koja stigne
    prvaPotvrda(...redniBrojevi) {
        return Promise.race(redniBrojevi.map(b => knjizenje(knjiga).potvrdi(b)));
    }
});

const nerealizovane = [...knjigaP].filter(x => x.status === "nerealizovana").map(x => x.redniBroj);
console.log("nerealizovane:", nerealizovane);

console.log("potvrda jedne:", (await knjizenje(knjigaP).potvrdi(nerealizovane[0])).opis);
try { await knjizenje(knjigaP).potvrdi(999); } catch (g) { console.log("otkaz:", g.message); }

const proknjizena = await knjizenje(knjigaP).proknjizi(...nerealizovane);
console.log("stanje pre knjiženja:", knjigaP.stanje, "| posle:", proknjizena.stanje);
console.log("statusi posle:", [...proknjizena].map(x => x.status));
console.log("original netaknut:", [...knjigaP].map(x => x.status));

console.log("race:", (await knjizenje(knjigaP).prvaPotvrda(...[...knjigaP].map(x => x.redniBroj))).opis);

try { await knjizenje(opozvana).proknjizi(...[...opozvana].map(x => x.redniBroj)); }
catch (g) { console.log("all pao (ima storniranih):", g.message); }

### Šta ovde donosi mehanizam sa K2

**`Promise` sa `setTimeout`.** Potvrda traje i može da otkaže — isti obrazac kao `dobaviArtikal` sa vežbi:
kašnjenje, otkaz za nepostojeći podatak, otkaz iz poslovnog razloga (stornirana transakcija).

**`Promise.all` bira „sve ili ništa".** Nova verzija knjige nastaje **tek pošto sve potvrde stignu**; ako jedna
otkaže, knjiga se uopšte ne menja. Time je nedeljivost knjiženja dobijena samim izborom metode, bez ručnog
poništavanja pola posla.

**`Promise.race` bira prvu potvrdu**, što je oblik za „javi čim bilo šta stigne".

**Sloj je odvojen od knjige.** `knjizenje(knjiga)` je funkcija koja oko postojeće knjige pravi asinhroni omotač,
pa `GlavnaKnjiga` ostaje ista kao u jedinici 3 — asinhronost nije ušla u tip, nego stoji pored njega.

---
# Jedinica 7 · Test i poređenje

In [ ]:
console.log("═══ isti zahtevi kao na K1 ═══");
const vega = ["Prodaja usluga|realizovana|u korist|2026-08-10|150000",
              "Plate|realizovana|na teret|2026-08-11|95000",
              "Reprezentacija|nerealizovana|na teret|2026-08-12|12000"]
    .map(red => red.split("|"))
    .reduce((k, [opis, status, tip, datum, iznos]) =>
        k.dodajTransakciju(Transakcija("170-9999", opis, status, datum, tip, Number(iznos))),
        GlavnaKnjiga("Vega ad", "20099887", "105566778"));

console.log("stanje:", vega.stanje);
console.log("transakcije (iterisanjem):", [...vega].map(x => `${x.redniBroj}:${x.opis}`));

const dodata = vega.dodajTransakciju(Transakcija("170-9999", "Kamata", "realizovana", "2026-08-13", "u korist", 4000));
console.log("posle dodavanja:", dodata.stanje, "| original:", vega.stanje, "| nova instanca:", dodata !== vega);
console.log("posle uklanjanja:", vega.ukloniTransakciju([...vega][1].redniBroj).stanje);

console.log("istorija:", istorija(dodata).map(k => k.stanje));
console.log("pretraga:", pretraga([realizovane, naTeret], dodata).map(x => x.opis));
console.log("opoziv:", [...opoziv(dodata, 2)].map(x => `${x.opis}(${x.status})`));

console.log("═══ dodatak: asinhrono knjiženje ═══");
const zaKnjizenje = [...dodata].filter(x => x.status === "nerealizovana").map(x => x.redniBroj);
const posle = await knjizenje(dodata).proknjizi(...zaKnjizenje);
console.log("stanje posle knjiženja:", posle.stanje, "| original:", dodata.stanje);

console.log("═══ lenjost generatora ═══");
console.log("dve najnovije verzije, ostatak lanca nije posećen:",
            uzmi(2, dodata.verzije()).map(k => k.stanje));
console.log("cela istorija kad zaista zatreba:", istorija(dodata).map(k => k.stanje));

### Šta se dobilo, a šta izgubilo

| | Dobitak | Cena |
| --- | --- | --- |
| generator za brojač | brojač je tok, potpuno zatvoren, zamenljiv drugim izvorom | `next().value` je duže od `n++` |
| iterabilna knjiga | `for...of`, spread, `yield*`; geter `transakcije` je sveden na `[...o]` | `[...knjiga]` u petlji lako pravi kvadratnu složenost |
| `verzije()` generator | lenj obilazak — ne gradi se ceo niz kad treba samo poslednjih par | za indeksiranje se ipak mora materijalizovati (`istorija`) |
| transdjuseri u `stanje` i `pretraga` | jedan prolaz bez obzira na broj kriterijuma; oblik rezultata se bira posebno | `compose(...kriterijumi.map(filter))` je teže za čitanje |
| `Promise` sloj | knjiženje „sve ili ništa" preko `all`, bez ručnog poništavanja | asinhronost se prenosi na sve pozivaoce (`await` do vrha) |

**Kada je koje rešenje bolje.** Za sam kolokvijum K1 osnovna verzija je brža za pisanje i lakša za odbranu.
Ova verzija se isplati kada podataka ima mnogo (jedan prolaz umesto `n`), kada se traži lenj pristup istoriji,
ili kada u zadatku stoji formulacija „minimalan broj iteracija", „omogućiti prolazak kroz kolekciju" ili
„dobavljanje traje neko vreme".